In [14]:
# Import required libraries for data loading and inspection
import pandas as pd
import os

In [15]:
# Set project root as a fixed absolute path
BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Project root:", BASE)
print("Exists:", os.path.exists(BASE))

Project root: /Users/alexia/Documents/CASA/Dissertation
Exists: True


## Load TS001 (usual residents)

In [20]:
# Load Census 2021 TS001: skip 7 Nomis metadata rows
ts001 = pd.read_csv(
    os.path.join(BASE, "03_data/demand/census2021/TS001_usual_residents_London_LSOA_2021.csv"),
    skiprows=7
)

# Keep only LSOA-level rows (Area contains 'lsoa2021:')
ts001 = ts001[ts001["Area"].str.contains("lsoa2021:", na=False)].copy()

# Extract LSOA code (e.g. E01011954) and LSOA name (e.g. Hartlepool 001A)
ts001["lsoa_code"] = ts001["Area"].str.extract(r"lsoa2021:(E\d+)")
ts001["lsoa_name"] = ts001["Area"].str.extract(r"E\d+ : (.+)$")

# Drop original Area column and rename population column
ts001 = ts001.drop(columns=["Area"])
ts001 = ts001.rename(columns={"2021": "total_residents"})

# Reorder columns
ts001 = ts001[["lsoa_code", "lsoa_name", "total_residents"]].reset_index(drop=True)

print("=== TS001 Usual Residents ===")
print("Shape:", ts001.shape)
print("Columns:", ts001.columns.tolist())
ts001.head()

=== TS001 Usual Residents ===
Shape: (35672, 3)
Columns: ['lsoa_code', 'lsoa_name', 'total_residents']


,lsoa_code,lsoa_name,total_residents
0,E01011954,Hartlepool 001A,2284
1,E01011969,Hartlepool 001B,1344
2,E01011970,Hartlepool 001C,1070
3,E01011971,Hartlepool 001D,1323
4,E01033465,Hartlepool 001F,1955


## Load TS045 (car/van availability)

In [21]:
# Peek at first 10 raw lines of TS045 to check Nomis metadata structure
with open(os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv"), encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()}")
        if i >= 9:
            break

Line 0: 
Line 1: "TS045 - Car or van availability"
Line 2: "ONS Crown Copyright Reserved [from Nomis on 26 May 2026]"
Line 3: "Population :","All households"
Line 4: "Units      :","Households"
Line 5: "Date       :","2021"
Line 6: 
Line 7: "Area","No cars or vans in household","1 car or van in household","2 cars or vans in household","3 or more cars or vans in household"
Line 8: 
Line 9: "gor:London",1440271,1380847,464970,137802


In [22]:
# Load Census 2021 TS045: skip 7 Nomis metadata rows
ts045 = pd.read_csv(
    os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv"),
    skiprows=7
)

# Keep only LSOA-level rows
ts045 = ts045[ts045["Area"].str.contains("lsoa2021:", na=False)].copy()

# Extract LSOA code and name
ts045["lsoa_code"] = ts045["Area"].str.extract(r"lsoa2021:(E\d+)")
ts045["lsoa_name"] = ts045["Area"].str.extract(r"E\d+ : (.+)$")

# Drop original Area column and rename car availability columns
ts045 = ts045.drop(columns=["Area"])
ts045 = ts045.rename(columns={
    "No cars or vans in household":            "cars_0",
    "1 car or van in household":               "cars_1",
    "2 cars or vans in household":             "cars_2",
    "3 or more cars or vans in household":     "cars_3plus"
})

# Reorder columns
ts045 = ts045[["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]].reset_index(drop=True)

print("=== TS045 Car/Van Availability ===")
print("Shape:", ts045.shape)
print("Columns:", ts045.columns.tolist())
ts045.head()

=== TS045 Car/Van Availability ===
Shape: (35672, 6)
Columns: ['lsoa_code', 'lsoa_name', 'cars_0', 'cars_1', 'cars_2', 'cars_3plus']


,lsoa_code,lsoa_name,cars_0,cars_1,cars_2,cars_3plus
0,E01011954,Hartlepool 001A,305,388.0,208.0,62.0
1,E01011969,Hartlepool 001B,87,296.0,168.0,53.0
2,E01011970,Hartlepool 001C,48,231.0,153.0,53.0
3,E01011971,Hartlepool 001D,29,201.0,204.0,86.0
4,E01033465,Hartlepool 001F,56,282.0,323.0,79.0


## Load IMD 2019 (deprivation index)

In [25]:
# Peek at IMD 2019 xlsx: check sheet names and first few rows
xl = pd.ExcelFile(os.path.join(BASE, "03_data/demand/imd2019/IMD2019_LSOA_England.xlsx"))
print("Sheet names:", xl.sheet_names)

# Preview first sheet
imd_raw = xl.parse(xl.sheet_names[0])
print("Shape:", imd_raw.shape)
print("Columns:", imd_raw.columns.tolist())
imd_raw.head()

Sheet names: ['Notes', 'IMD2019']
Shape: (0, 0)
Columns: []


""


In [26]:
# Load IMD 2019 from the correct sheet 'IMD2019'
imd_raw = xl.parse("IMD2019")
print("Shape:", imd_raw.shape)
print("Columns:", imd_raw.columns.tolist())
imd_raw.head()

Shape: (32844, 6)
Columns: ['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Rank', 'Index of Multiple Deprivation (IMD) Decile']


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Rank,Index of Multiple Deprivation (IMD) Decile
0,E01000001,City of London 001A,E09000001,City of London,29199,9
1,E01000002,City of London 001B,E09000001,City of London,30379,10
2,E01000003,City of London 001C,E09000001,City of London,14915,5
3,E01000005,City of London 001E,E09000001,City of London,8678,3
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,14486,5


In [27]:
# Load IMD 2019 from 'IMD2019' sheet: contains deprivation rank and decile per LSOA (2011 boundaries)
imd = xl.parse("IMD2019")

# Rename columns to concise snake_case names
imd = imd.rename(columns={
    "LSOA code (2011)":                          "lsoa_code_2011",
    "LSOA name (2011)":                          "lsoa_name",
    "Local Authority District code (2019)":      "lad_code",
    "Local Authority District name (2019)":      "lad_name",
    "Index of Multiple Deprivation (IMD) Rank":  "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile":"imd_decile"
})

print("=== IMD 2019 ===")
print("Shape:", imd.shape)
print("Columns:", imd.columns.tolist())
imd.head()

=== IMD 2019 ===
Shape: (32844, 6)
Columns: ['lsoa_code_2011', 'lsoa_name', 'lad_code', 'lad_name', 'imd_rank', 'imd_decile']


,lsoa_code_2011,lsoa_name,lad_code,lad_name,imd_rank,imd_decile
0,E01000001,City of London 001A,E09000001,City of London,29199,9
1,E01000002,City of London 001B,E09000001,City of London,30379,10
2,E01000003,City of London 001C,E09000001,City of London,14915,5
3,E01000005,City of London 001E,E09000001,City of London,8678,3
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,14486,5


##  Load OpenStreetEV GLA

In [35]:
# Load restricted OpenStreetEV GLA dataset
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))

# Keep only columns with meaningful content; drop mostly-NaN operator/owner/media cols
cols_keep = ["country_co", "id", "external_l", "name", "address",
             "postal_cod", "state", "coordinate", "coordina_1",
             "location_c", "location_1"]
osev = osev_raw[cols_keep].copy()

# Rename truncated column names to full descriptive names
osev = osev.rename(columns={
    "country_co": "country_code",
    "id":         "location_id",
    "external_l": "external_uuid",
    "name":       "location_name",
    "address":    "address",
    "postal_cod": "postcode",
    "state":      "borough",
    "coordinate": "latitude",
    "coordina_1": "longitude",
    "location_c": "location_category",
    "location_1": "location_type",
})

print("=== OpenStreetEV GLA ===")
print("Shape:", osev.shape)
print("Columns:", osev.columns.tolist())
osev.head()

=== OpenStreetEV GLA ===
Shape: (23045, 11)
Columns: ['country_code', 'location_id', 'external_uuid', 'location_name', 'address', 'postcode', 'borough', 'latitude', 'longitude', 'location_category', 'location_type']


,country_code,location_id,external_uuid,location_name,address,postcode,borough,latitude,longitude,location_category,location_type
0,GB,3SJEP0S,317c299b-f2f3-5919-a2ed-e8d34cc38036,572 Rainham Road South,Dagenham,RM10 7XD,Barking and Dagenham,51.546391,0.165086,On-Street,On-street
1,GB,DGGPFSM,e958b4bf-e121-55fa-a7cb-638e6e09e7cd,511 Gale Street,Dagenham,RM9 4TP,Barking and Dagenham,51.539635,0.127792,On-Street,On-street
2,GB,5ZGAF24,5ZGAF24,Lidl Dagenham,807-829 Longbridge Road,RM8 2DB,Barking and Dagenham,51.552335,0.113204,Supermarket,Destination
3,GB,9E81MQO,9E81MQO,Lidl Beacontree Heath,79 Whalebone Lane,RM8 1AJ,Barking and Dagenham,51.563726,0.146107,Supermarket,Destination
4,GB,21CK7GS,BE12C28,Lidl Dagenham Heathway,258-262 Heathway,RM10 8QS,Barking and Dagenham,51.542202,0.148508,Supermarket,Destination


## Load GLA location–EVSE join

In [29]:
# Load restricted dataset: join table linking charger locations to individual EVSE units
gla_join = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/gla_location_evse_join.csv"))
print("=== GLA Location-EVSE Join ===")
print("Shape:", gla_join.shape)
print("Columns:", gla_join.columns.tolist())
gla_join.head()

=== GLA Location-EVSE Join ===
Shape: (38514, 2)
Columns: ['location_id', 'evse_id']


,location_id,evse_id
0,TSM91W3,174b834d444d9c65929b489ce7e76fb6
1,TSM91W3,e75ec419ba4f4e378994795885a17595
2,TSM91W3,5c2d9027b4d343139d5cce55f45193ce
3,QUKDP0S,4f99a28a02d9c920e711e599248fbd82
4,QUKDP0S,cb134f1501e04f6290b201056cde1126


## Load EVSE status

In [30]:
# Load restricted dataset: operational status records for each EVSE unit in GLA
evse_status = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/evse_status_gla.csv"))
print("=== EVSE Status GLA ===")
print("Shape:", evse_status.shape)
print("Columns:", evse_status.columns.tolist())
evse_status.head()

=== EVSE Status GLA ===
Shape: (25461562, 4)
Columns: ['uid', 'evse_uid', 'last_updated', 'status']


,uid,evse_uid,last_updated,status
0,o-1765736769,0002271b5e174a428d01e9847b69d195,2025-09-02 10:49:21,AVAILABLE
1,o-1795225869,0003eb1b04bbe87d2a4bbd95ea85fbde,2025-09-19 07:05:48,AVAILABLE
2,c-1732589905,0007804415dd5bf2c01750f4bf0c4938,2025-08-06 04:42:54,AVAILABLE
3,o-1339343093,001255f66bcb442fa81d8e7fdbec81ab,2025-01-17 12:13:32,AVAILABLE
4,o-1416124652,001255f66bcb442fa81d8e7fdbec81ab,2025-03-04 05:19:06,AVAILABLE


## Load device all UK

In [31]:
# Load restricted dataset: all registered EV charging devices across the UK (large file)
device = pd.read_csv(
    os.path.join(BASE, "03_data/restricted/gla/device_all_uk.csv"),
    low_memory=False
)
print("=== Device All UK ===")
print("Shape:", device.shape)
print("Columns:", device.columns.tolist())
device.head()

=== Device All UK ===
Shape: (133263, 4)
Columns: ['location_id', 'zapmap_device_uid', 'power_band', 'device_max_power']


,location_id,zapmap_device_uid,power_band,device_max_power
0,6SO91US,98ANCHF,1. Slow,2760
1,LOPDTFA,QYY5JSI,1. Slow,2760
2,UTC0HJS,2VEZNZJ,1. Slow,2760
3,B4QJ4NO,DLP3M40,1. Slow,2990
4,Y9RMDL3,YEU1W4B,1. Slow,2990


## OpenStreetEV Check full column names

In [32]:
# Check full column names before truncation — read with no display limit
import pandas as pd
pd.set_option("display.max_colwidth", None)
print(osev.columns.tolist())

['country_co', 'party_id', 'Source', 'id', 'external_l', 'name', 'address', 'city', 'postal_cod', 'state', 'country', 'related_lo', 'coordinate', 'coordina_1', 'parking_ty', 'directions', 'operator_n', 'operator_w', 'operator_l', 'suboperato', 'subopera_1', 'subopera_2', 'owner_name', 'owner_webs', 'owner_logo', 'opening_ti', 'charging_w', 'images', 'energy_mix', 'last_updat', 'created_at', 'deleted_at', 'location_c', 'location_1']


In [33]:
# Inspect first 2 rows to infer what each truncated column contains
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
osev.head(2).T  # .T transposes so each column shows as a row — easier to read 34 columns

,0,1
country_co,GB,GB
party_id,NaN,NaN
Source,Custom,Custom
id,3SJEP0S,DGGPFSM
external_l,317c299b-f2f3-5919-a2ed-e8d34cc38036,e958b4bf-e121-55fa-a7cb-638e6e09e7cd
name,572 Rainham Road South,511 Gale Street
address,Dagenham,Dagenham
city,Dagenham,Dagenham
postal_cod,RM10 7XD,RM9 4TP
state,Barking and Dagenham,Barking and Dagenham


## Summary check

In [36]:
# Print a summary table to confirm all 7 datasets loaded successfully
datasets = {
    "TS001":        ts001,
    "TS045":        ts045,
    "IMD2019":      imd,
    "OpenStreetEV": osev,
    "GLA_Join":     gla_join,
    "EVSE_Status":  evse_status,
    "Device_UK":    device,
}

print(f"{'Dataset':<15} {'Rows':>10} {'Cols':>6}")
print("-" * 35)
for name, df in datasets.items():
    print(f"{name:<15} {df.shape[0]:>10,} {df.shape[1]:>6}")

Dataset               Rows   Cols
-----------------------------------
TS001               35,672      3
TS045               35,672      6
IMD2019             32,844      6
OpenStreetEV        23,045     11
GLA_Join            38,514      2
EVSE_Status     25,461,562      4
Device_UK          133,263      4


TS001 + TS045 — 筛选出 London LSOA，float 转 int，计算总车辆数和有车家庭比例

IMD 2019 — 筛选 London LSOA，处理 2011→2021 LSOA code 变化问题

OpenStreetEV — 检查坐标是否在 London 范围内，检查重复 location_id

EVSE Status — 从 2500 万行聚合成每个 EVSE 的可用率

Device UK — 筛选出 GLA 范围内的设备